## Introduction

Airline ticket prices vary greatly depending on a number of factors such as travel distance, time of booking, airline, and more. Understanding what influences these prices can help travelers make better decisions and help airlines with pricing strategies.

In this project, I aim to build a regression model to predict airline ticket prices using flight itinerary data. I will start by cleaning and filtering the data, then select relevant features to train a model. The goal is to find patterns that help explain variations in ticket pricing and evaluate how accurately we can predict those prices.

The dataset used in this project is over 31 GB in size and contains millions of rows of flight itinerary information. Due to its size and complexity, I will be working with the data in chunks and filtering it down to a manageable sample in order to build and test my model effectively.

## Data Cleaning and Preprocessing

Since the original dataset is over 31 GB, I could not load it into memory all at once. To handle this, I used chunking with pandas to read the file in smaller parts. I also filtered the data to include only rows where the airline was "American Airlines" to reduce the size further and narrow the focus of the analysis.

After filtering, I inspected the resulting dataset to check for missing values, unusual data types, and overall structure. I then created a smaller sample of 500 rows to use for model development. This allowed me to test preprocessing steps and train a model without overloading the system.

Key steps taken:
- Used `read_csv()` with `chunksize=100,000` to safely read the large file
- Filtered rows where `'segmentsAirlineName' == 'American Airlines'`
- Dropped or filled missing values where needed
- Confirmed data types and column names
- Selected relevant features for modeling

The cleaned dataset includes variables such as `totalFare`, `elapsedDays`, and `totalTravelDistance`, which will be used in the modeling stage.



In [35]:
import pandas as pd
import numpy as np


In [36]:
file_path = "/Users/orlandobrandon/Desktop/Portfolio Files/orlandobrandon.github.io/Project_3/itineraries.csv"


In [37]:
sample = pd.read_csv(file_path, nrows=5)
print(sample.columns)


Index(['legId', 'searchDate', 'flightDate', 'startingAirport',
       'destinationAirport', 'fareBasisCode', 'travelDuration', 'elapsedDays',
       'isBasicEconomy', 'isRefundable', 'isNonStop', 'baseFare', 'totalFare',
       'seatsRemaining', 'totalTravelDistance',
       'segmentsDepartureTimeEpochSeconds', 'segmentsDepartureTimeRaw',
       'segmentsArrivalTimeEpochSeconds', 'segmentsArrivalTimeRaw',
       'segmentsArrivalAirportCode', 'segmentsDepartureAirportCode',
       'segmentsAirlineName', 'segmentsAirlineCode',
       'segmentsEquipmentDescription', 'segmentsDurationInSeconds',
       'segmentsDistance', 'segmentsCabinCode'],
      dtype='object')


In [38]:
filtered_chunks = []
max_rows = 500
row_total = 0
chunk_num = 0

for chunk in pd.read_csv(file_path, chunksize=100_000):
    chunk_num += 1
    filtered = chunk[chunk['segmentsAirlineName'] == 'American Airlines']
    row_total += len(filtered)

    print(f"Chunk {chunk_num}: found {len(filtered)} rows — Total so far: {row_total}")

    if not filtered.empty:
        filtered_chunks.append(filtered)

    if row_total >= max_rows:
        break

filtered_df = pd.concat(filtered_chunks, ignore_index=True)


Chunk 1: found 9186 rows — Total so far: 9186


In [39]:
print(filtered_df.shape)
print(filtered_df.head())
print(filtered_df.isnull().sum())
print(filtered_df.dtypes)


(9186, 27)
                              legId  searchDate  flightDate startingAirport  \
0  e1b95e4e6c997517f64ba5ea712bda22  2022-04-16  2022-04-17             ATL   
1  68063b78f62c81ac88820015908ac208  2022-04-16  2022-04-17             ATL   
2  33dba4888a3a68195d53ec2cc8b0a2d6  2022-04-16  2022-04-17             ATL   
3  99666577c217e0a6b54a17e2f8ab9a7b  2022-04-16  2022-04-17             ATL   
4  dc2eda19b0499170f1f7f8c5a179ac07  2022-04-16  2022-04-17             ATL   

  destinationAirport fareBasisCode travelDuration  elapsedDays  \
0                BOS      KH0AUEY5        PT2H38M            0   
1                BOS      HH0KUEY5        PT2H44M            0   
2                CLT      M0AHZNN1         PT1H8M            0   
3                CLT      M0AHZNN1        PT1H21M            0   
4                CLT      M0AHZNN1        PT1H22M            0   

   isBasicEconomy  isRefundable  ...  segmentsArrivalTimeEpochSeconds  \
0           False         False  ...        

In [40]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Define features and target
X = filtered_df[['elapsedDays', 'totalTravelDistance']]
y = filtered_df['totalFare']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and train the model
model = LinearRegression()
model.fit(X_train, y_train)

# Predict on test set
y_pred = model.predict(X_test)

# Evaluate the model
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse:.2f}")
print(f"R-squared Score: {r2:.2f}")


Mean Squared Error: 85537.91
R-squared Score: 0.06


## Linear Regression

To build the regression model, I selected `elapsedDays` and `totalTravelDistance` as features to predict `totalFare`, the price of the airline ticket.

I split the dataset into training and testing sets, trained a linear regression model, and then evaluated its performance using Mean Squared Error (MSE) and R-squared (R²).

This first experiment helps establish a baseline model. The R-squared score tells how much of the variation in ticket prices can be explained by the two features. In future steps, I will explore additional features and model types to improve performance.
